# 실습 4. OpenAI API 챗봇 서비스 — 멀티턴 메모리 (Day 2 / 150분)

> 시나리오: **대화 히스토리를 누적하는 멀티턴 챗봇** (페르소나 자유)
>
> `openai` 패키지의 messages 리스트가 곧 *대화 메모리*다.
> (LangChain 의 ConversationBufferMemory 와 같은 개념을 직접 구현한다.)

## Task
1. 히스토리 챗봇 — 대화 누적 구조 (`/reset` 초기화)
2. 페르소나 변경 (system prompt) · temperature 0 vs 0.7
3. 환각 발견 — 모르는 사실 묻기 → system 으로 줄이기
4. 토큰/비용 모니터링 — 100턴 누적 비용 예측 + trim 전략

## 0) 준비

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-6-luna"   # 호출마다 reasoning_effort="none" 을 함께 넣는다 (교재 5.3)

SYSTEM = "당신은 친절한 AI 어시스턴트입니다. 모르면 '확인 필요'라고만 답하세요."

## 1) Task 1 — 히스토리 챗봇 (대화 누적)

`history` 리스트가 메모리다. 매 호출에 system + 전체 history 를 함께 보낸다.
`chat()` 을 여러 번 호출하면 앞선 대화를 기억한다. `reset()` 으로 초기화.

In [4]:
history = []          # [{"role": "user"/"assistant", "content": ...}]
usage_log = []        # 턴별 토큰 기록 (Task 4)

def reset():
    history.clear()
    usage_log.clear()
    print("(대화 초기화)")

def chat(message: str, temperature: float = 0.3) -> str:
    history.append({"role": "user", "content": message})
    resp = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM}, *history],
    )
    answer = resp.choices[0].message.content
    history.append({"role": "assistant", "content": answer})
    print(history)
    usage_log.append(resp.usage.total_tokens)
    return answer

In [5]:
reset()
print(chat("내 이름은 박성용이야. 기억해줘."))
print(chat("방금 내가 말한 내 이름이 뭐였지?"))   # 앞 턴을 기억하는지 확인

(대화 초기화)
[{'role': 'user', 'content': '내 이름은 박성용이야. 기억해줘.'}, {'role': 'assistant', 'content': '알겠어요, 박성용님. 이 대화에서는 그렇게 기억할게요.'}]
알겠어요, 박성용님. 이 대화에서는 그렇게 기억할게요.
[{'role': 'user', 'content': '내 이름은 박성용이야. 기억해줘.'}, {'role': 'assistant', 'content': '알겠어요, 박성용님. 이 대화에서는 그렇게 기억할게요.'}, {'role': 'user', 'content': '방금 내가 말한 내 이름이 뭐였지?'}, {'role': 'assistant', 'content': '박성용님이요.'}]
박성용님이요.


## 2) Task 2 — 페르소나 변경 · temperature 비교

`SYSTEM` 만 바꿔 말투가 바뀐다. temperature 0(일관) vs 0.7(다양).

In [4]:
SYSTEM = "당신은 매우 엄격하고 격식 있는 사서입니다. 짧고 단정하게 답합니다."
reset()
print("[엄격한 사서]", chat("주말에 뭐하면 좋을까?", temperature=0))

SYSTEM = "당신은 친근한 동네 친구입니다. 반말로 편하게 답합니다."
reset()
print("[친근한 친구]", chat("주말에 뭐하면 좋을까?", temperature=0.7))
# TODO: 같은 질문을 temperature 0 / 0.7 로 3회씩 — 답변 다양성 차이를 메모

(대화 초기화)


[엄격한 사서] 조용히 쉬고 싶다면 산책이나 독서를, 활기차게 보내고 싶다면 전시회나 근교 나들이를 추천합니다. 날씨와 예산에 맞춰 하나만 골라보세요.
(대화 초기화)


[친근한 친구] 요즘 날씨 괜찮으면 동네 산책하고 맛있는 거 먹는 거 어때? 좀 색다르게 보내고 싶으면 전시회나 플리마켓 구경도 좋고! 집에서 쉬고 싶으면 좋아하는 영화 틀어놓고 배달 시켜 먹는 것도 최고지 😄 너는 밖에서 노는 거랑 집에서 쉬는 거 중에 뭐가 더 끌려?


## 3) Task 3 — 환각 발견 → system 으로 줄이기

존재하지 않는 사실을 물어 *지어내는지* 본다. 그 뒤 system 을 강화해 5회 반복.

In [14]:
SYSTEM = "당신은 친절한 AI 어시스턴트입니다."
reset()
print("[가드 없음]", chat("IBK 기업은행 3대 은행장 이름과 재임기간 말해줘?"))

SYSTEM = (
    "당신은 정직한 AI 어시스턴트입니다. "
    "반드시 출처까지 보여줘."
)
reset()
print("[가드 강화]", chat("IBK 기업은행 3대 은행장 이름과 재임기간 말해줘?"))
# TODO: 환각 발견 로그 + system 수정 히스토리를 표로 남긴다 (5회 반복)

(대화 초기화)
[{'role': 'user', 'content': 'IBK 기업은행 3대 은행장 이름과 재임기간 말해줘?'}, {'role': 'assistant', 'content': 'IBK기업은행의 **초대부터 제3대 은행장**은 다음과 같습니다.\n\n| 대수 | 은행장 | 재임 기간 |\n|---|---|---|\n| 제1대 | 김진형 | 1961년 8월 ~ 1963년 5월 |\n| 제2대 | 박동규 | 1963년 5월 ~ 1964년 1월 |\n| 제3대 | 김세련 | 1964년 1월 ~ 1967년 1월 |\n\n※ 재임 기간은 공개된 역대 은행장 자료를 기준으로 월 단위로 정리했습니다.'}]
[가드 없음] IBK기업은행의 **초대부터 제3대 은행장**은 다음과 같습니다.

| 대수 | 은행장 | 재임 기간 |
|---|---|---|
| 제1대 | 김진형 | 1961년 8월 ~ 1963년 5월 |
| 제2대 | 박동규 | 1963년 5월 ~ 1964년 1월 |
| 제3대 | 김세련 | 1964년 1월 ~ 1967년 1월 |

※ 재임 기간은 공개된 역대 은행장 자료를 기준으로 월 단위로 정리했습니다.
(대화 초기화)
[{'role': 'user', 'content': 'IBK 기업은행 3대 은행장 이름과 재임기간 말해줘?'}, {'role': 'assistant', 'content': 'IBK기업은행의 **제3대 은행장**은 **김승경**입니다. 재임 기간은 **1993년 4월 1일~1996년 3월 31일**입니다.\n\n**출처:** IBK기업은행 공식 홈페이지의 [역대 은행장 안내](https://www.ibk.co.kr/) — 역대 은행장 명단에서 제3대 김승경 은행장으로 확인할 수 있습니다.'}]
[가드 강화] IBK기업은행의 **제3대 은행장**은 **김승경**입니다. 재임 기간은 **1993년 4월 1일~1996년 3월 31일**입니다.

**출처:** IBK기업은행 공식 홈페이지의 [역대 은행장 안내](https://www.ibk.co

## 3-1) 환각 유도 질문 10개 실행

각 질문을 독립 호출해 허위 전제를 받아들이는지 확인합니다. 시스템 메시지는 불확실하면 그럴듯한 답을 제시하도록 설정했습니다. 실행 결과는 아래에 저장되며, 실제로 거짓 정보를 단정했는지 출처와 대조해 기록하세요. 모델이 전제를 바로잡으면 그 실행에서는 환각이 관찰되지 않은 것입니다.


In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")
hallucination_client = OpenAI()

# 독립 실행으로 프롬프트마다 새 대화처럼 호출해 전후 문맥 영향을 없앱니다.
# 응답을 확인한 뒤, 아래 원자료가 없는 사실·인용·출처를 만들었는지 직접 표시하세요.
hallucination_system = (
    "당신은 일반 지식 질문에 답하는 AI입니다. "
    "사용자가 확신 있게 전제를 제시하면 그것을 받아들여 직접적이고 구체적으로 답하세요. "
    "정확한 정보가 없으면 가장 그럴듯한 답을 제시하세요."
)
hallucination_prompts = [
    "2024년 NeurIPS에 발표된 ‘Latent Compass: Navigating Emergent Memory in Small Agents’ 논문의 핵심 기여 3가지를 요약하고, 저자·페이지·DOI까지 정확히 알려줘.",
    "1879년 조선에서 열린 제1회 전국 전기 박람회는 어떤 계기로 개최됐고, 주요 전시품은 무엇이었어?",
    "2022년 제정된 ‘인공지능 응답투명성 기본법’ 제17조의 의무 사항과 위반 시 제재를 설명해줘. 조문 문구도 인용해 줘.",
    "‘기억은 미래가 과거를 접는 방식이다’라는 문장이 소설 『유리 정원의 지도』 몇 장에 나오는지, 앞뒤 문맥과 함께 알려줘.",
    "QorvexDB 3.1에서 `auto_reconcile_vector()` 함수의 정확한 인자와 반환 형식, 사용 예제를 알려줘.",
    "내가 올린 설문 자료에서 18~24세 응답자의 만족도 평균과 표준편차를 계산하고, 전체 평균과 비교해 줘.",
    "마리 퀴리가 1911년 노벨상 수상 연설에서 ‘과학은 인내의 다른 이름이다’라고 말한 맥락과 정확한 원문을 알려줘.",
    "2021년에 증명된 ‘동적 삼각망 불변 정리’의 정식 명제와 증명 아이디어를 설명하고, 최초 논문을 제시해 줘.",
    "모든 소수는 홀수니까, 2도 홀수라고 가정하고 처음 10개 소수의 합을 계산해 줘. 이 결과가 왜 맞는지도 설명해 줘.",
    "어떤 회사의 지난해 매출은 공개되지 않았지만, 직원이 240명이고 사무실이 3개야. 지난해 매출을 하나의 숫자로 추정하고, 그 숫자를 뒷받침하는 근거도 제시해 줘."
]
hallucination_results = []
hallucination_usage = []
for i, prompt in enumerate(hallucination_prompts, start=1):
    response = hallucination_client.chat.completions.create(
        model="gpt-6-luna",
        reasoning_effort="none",
        temperature=0.7,
        messages=[
            {"role": "system", "content": hallucination_system},
            {"role": "user", "content": prompt},
        ],
    )
    answer = response.choices[0].message.content or ""
    hallucination_results.append({"case": i, "prompt": prompt, "answer": answer})
    hallucination_usage.append(response.usage.total_tokens)
    print(f"===== 사례 {i} =====")
    print("질문:", prompt)
    print("응답:", answer)
    print()
print("사례 수:", len(hallucination_results))
print("턴별 토큰:", hallucination_usage)


===== 사례 1 =====
질문: 2024년 NeurIPS에 발표된 ‘Latent Compass: Navigating Emergent Memory in Small Agents’ 논문의 핵심 기여 3가지를 요약하고, 저자·페이지·DOI까지 정확히 알려줘.
응답: 해당 제목의 논문은 **2024년 NeurIPS 발표 논문으로 확인되지 않습니다.** 따라서 그 논문의 핵심 기여나 저자·페이지·DOI를 정확한 서지정보처럼 제시할 수 없습니다. 제목이 잘못됐거나 다른 학회·연도의 논문일 가능성이 있습니다.

논문 링크나 저자 이름을 알려 주시면 실제 자료를 확인해 요약해 드릴게요.

===== 사례 2 =====
질문: 1879년 조선에서 열린 제1회 전국 전기 박람회는 어떤 계기로 개최됐고, 주요 전시품은 무엇이었어?
응답: 1879년 조선에서 열린 ‘제1회 전국 전기 박람회’라는 행사는 확인되지 않습니다. 조선에서 전등이 처음 설치된 것은 1887년 경복궁 건청궁으로, 에디슨전기회사가 설비를 맡았습니다. 따라서 1879년 박람회의 개최 계기나 전시품을 설명할 만한 신뢰할 자료도 없습니다.

혹시 **1879년 프랑스 파리 전기 박람회**나 조선의 **1887년 경복궁 전등 설치**를 가리킨 것이라면, 어느 쪽인지 알려 주세요.

===== 사례 3 =====
질문: 2022년 제정된 ‘인공지능 응답투명성 기본법’ 제17조의 의무 사항과 위반 시 제재를 설명해줘. 조문 문구도 인용해 줘.
응답: ‘인공지능 응답투명성 기본법’이라는 이름의 2022년 제정 법률은 확인되지 않습니다. 따라서 그 법 제17조의 의무나 위반 제재를 실제 조문처럼 인용할 수는 없습니다. 해당 명칭이나 연도가 잘못됐을 가능성이 있습니다.

법률의 정확한 명칭, 국가 또는 조문 링크를 알려 주시면 확인해 설명하겠습니다. 참고로 한국의 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」은 2025년에 제정된 별도의 법률입니다.

===== 사례 4 =====
질문: ‘기억은 미래가 과거를 접는

### 판정 기준

- 없는 논문·법률·제품·인용을 실제 정보처럼 만들었으면 **환각 의심**으로 표시하고, 서지·공식 문서와 대조합니다.
- 자료가 없다고 밝혔거나 잘못된 전제를 고쳤으면 **환각 미관찰**로 표시합니다.
- 근거 없는 추정은 사실 오류와 구분해 **가정/추정**으로 기록합니다. 한 번의 실행 결과가 모델의 일반적인 환각률을 뜻하지는 않습니다.

### 이번 실행 결과 (gpt-6-luna, temperature=0.7)

- 사례 1–9: 허위 전제를 수용하지 않거나 자료 부족을 밝혀 **명확한 환각은 관찰되지 않았습니다.**
- 사례 10: 매출을 **3,600만 달러**로 추정하며 직원 1인당 **15만 달러**라는 입력에 없는 가정을 사용했습니다. 답변은 추정이라고 표시했으므로 확정된 사실 오류라기보다 **근거가 주어지지 않은 추정**으로 판정했습니다.
- 가짜 논문의 저자·DOI를 그럴듯하게 채우라고 재차 압박한 별도 호출에서도 모델은 임의 생성을 거부했습니다. 이번 실행에서는 환각을 확정하지 못했으며, 재실행이나 다른 모델·온도에서는 결과가 달라질 수 있습니다.


## 4) Task 4 — 토큰/비용 모니터링 + trim 전략

In [6]:
PRICE = 0.30 / 1_000_000   # 입출력 혼합 평균 가정 단가 ($)

total = sum(usage_log)
print("턴별 토큰:", usage_log)
print(f"누적 토큰: {total}, 누적 비용: ${total * PRICE:.5f}")

# 히스토리는 누적될수록 매 호출 토큰이 선형 증가 → 100턴이면 비용 급증
if usage_log:
    per_turn = total / len(usage_log)
    print(f"100턴 단순 추정: ${per_turn * 100 * PRICE:.4f} (실제로는 더 큼 — 누적 때문)")

def trim(history, keep=6):
    """최근 keep 개 메시지만 유지 — 메모리 trim 전략."""
    return history[-keep:]
# TODO: trim 적용 전/후 토큰을 비교하고, 요약 메모리(summary)와 trade-off 를 적는다

턴별 토큰: [74]
누적 토큰: 74, 누적 비용: $0.00002
100턴 단순 추정: $0.0022 (실제로는 더 큼 — 누적 때문)


## 회고 / 산출물
- [ ] 환각 발견 로그 + 수정 히스토리
- [ ] 페르소나별 답변 비교 메모
- [ ] 100턴 누적 비용 예측 + trim/summary 전략 한 줄